In [1]:
import Pkg
Pkg.activate("../chebcoefs")
Pkg.instantiate()

  Activating project at `~/repos/WaveGreen2D/chebcoefs`


In [2]:
using StaticArrays
using FastChebInterp

include("Chebyshev.jl")
using .Chebyshev
import .Chebyshev: normalize, contains, clenshaw, gradient_clenshaw, hessian_clenshaw

In [3]:
f(x) = sin(x)
lb, ub = SA[0.0], SA[0.5π]
xc = chebpoints(50, lb[], ub[])
cb = chebinterp(f.(xc), lb[], ub[]) 
cs = ChebyshevSeries(cb.coefs, lb, ub)
print(cs)
cs

1-D Chebyshev series of order 13

1-dimensional Chebyshev series of order 13 for x ∈ [0.000, 1.571]

In [4]:
f(x) = cos(0.25*x[1]*x[2])
lb = SA[-0.05, 0.2]
ub = SA[0.15, 0.4]
xc = chebpoints((50, 50), lb, ub)
cb = chebinterp(f.(xc), lb, ub)
cs = ChebyshevSeries(cb.coefs, lb, ub)
print(cs)
cs

2-D Chebyshev series of order (4, 4)

2-dimensional Chebyshev series of order (4, 4) for x ∈ [-0.050, 0.150]×[0.200, 0.400]

In [5]:
f(x) = exp(x[1]*x[2]) * cos(x[1] + x[3]/2)
lb = SA[0.5, -0.2, 1.0]
ub = SA[0.7, 0.0, 1.2]
xc = chebpoints((50, 50, 50), lb, ub)
cb = chebinterp(f.(xc), lb, ub)
cs = ChebyshevSeries(cb.coefs, lb, ub)
print(cs)
cs

3-D Chebyshev series of order (8, 7, 7)

3-dimensional Chebyshev series of order (8, 7, 7) for x ∈ [0.500, 0.700]×[-0.200, 0.000]×[1.000, 1.200]

In [6]:
f(u) = sin(u^2)
u(x::SVector{1,Float64}) = x .^ 0.5
∇u(x::SVector{1,Float64}) = reshape(0.5 * x .^ -0.5, Size(1, 1))
Hu(x::SVector{1,Float64}) = reshape(-0.25 * x .^ -1.5, Size(1, 1, 1))

lb, ub = SA[0.0], SA[1.5]
uc = chebpoints(50, lb[], ub[])
cb = chebinterp(f.(uc), lb[], ub[]) 
cs = ChebyshevSeries(cb.coefs, lb, ub)
ts = TransformedChebyshevSeries(cs, u, ∇u, Hu)
print(ts)
ts

1-D transformed Chebyshev series of order 22

1-dimensional transformed Chebyshev series of order 22 for u(x) ∈ [0.000, 1.500]

In [7]:
function f(u::SVector{2, Float64})
    r, θ = u
    return exp(r*cos(θ)) * cos(r*sin(θ))
end

function u(x::SVector{2,Float64})
    ξ, η = x

    r = sqrt(ξ^2 + η^2)
    θ = atan(η / ξ)

    return SA[r, θ]
end

function ∇u(x::SVector{2,Float64})
    ξ, η = x

    r² = ξ^2 + η^2
    r = √r²

    r₁ = ξ / r
    r₂ = η / r
    θ₁ = -η / r²
    θ₂ = ξ / r²

    ∇u₁ = SA[r₁, r₂]
    ∇u₂ = SA[θ₁, θ₂]

    return vcat(∇u₁', ∇u₂')
end

function Hu(x::SVector{2,Float64})
    ξ, η = x

    ξ², η² = ξ^2, η^2

    r² = ξ² + η²
    r = √r²
    r³ = r² * r
    r⁴ = r³ * r

    r₁₁ = η² / r³
    r₁₂ = -ξ * η / r³
    r₂₁ = r₁₂
    r₂₂ = ξ² / r³

    θ₁₁ = 2 * ξ * η / r⁴
    θ₁₂ = (η² - ξ²) / r⁴
    θ₂₁ = θ₁₂
    θ₂₂ = -2 * ξ * η / r⁴

    Hu₁ = reshape([r₁₁ r₁₂;
            r₂₁ r₂₂], 1, 2, 2)

    Hu₂ = reshape([θ₁₁ θ₁₂;
            θ₂₁ θ₂₂], 1, 2, 2)

    hess_u = vcat(Hu₁, Hu₂)

    return SArray{Tuple{2,2,2},Float64}(hess_u)
end

lb = SA[0.1, -0.7]
ub = SA[2.0, 1.3]
uc = chebpoints((50, 50), lb, ub)
cb = chebinterp(f.(uc), lb, ub)
cs = ChebyshevSeries(cb.coefs, lb, ub)
ts = TransformedChebyshevSeries(cs, u, ∇u, Hu)
print(ts)
ts

2-D transformed Chebyshev series of order (14, 31)

2-dimensional transformed Chebyshev series of order (14, 31) for u(x) ∈ [0.100, 2.000]×[-0.700, 1.300]

In [8]:
function f(u::SVector{3, Float64})
    r, θ, ϕ = u
    r² = r^2
    return r² * sin(ϕ) * cos(θ) * cos(ϕ) * exp(-r²)
end

function u(x::SVector{3,Float64})
    ξ, η, ζ = x

    ρ = sqrt(ξ^2 + η^2)

    r = sqrt(ξ^2 + η^2 + ζ^2)
    θ = atan(η / ξ)
    ϕ = atan(ρ / ζ)

    return SA[r, θ, ϕ]
end

function ∇u(x::SVector{3,Float64})
    ξ, η, ζ = x

    r² = ξ^2 + η^2 + ζ^2
    r = √r²
    ρ² = ξ^2 + η^2
    ρ = √ρ²

    r₁ = ξ / r
    r₂ = η / r
    r₃ = ζ / r
    θ₁ = -η / ρ²
    θ₂ = ξ / ρ²
    θ₃ = 0
    ϕ₁ = ξ * ζ / (ρ * r²)
    ϕ₂ = η * ζ / (ρ * r²)
    ϕ₃ = -ρ / r²

    ∇u₁ = SA[r₁, r₂, r₃]
    ∇u₂ = SA[θ₁, θ₂, θ₃]
    ∇u₃ = SA[ϕ₁, ϕ₂, ϕ₃]

    return vcat(∇u₁', ∇u₂', ∇u₃')
end

function Hu(x::SVector{3,Float64})
    ξ, η, ζ = x

    ξ² = ξ^2
    η² = η^2
    ζ² = ζ^2

    r² = ξ² + η² + ζ²
    r = √r²
    r³ = r² * r
    r⁴ = r³ * r

    ρ² = ξ² + η²
    ρ = √ρ²
    ρ³ = ρ² * ρ
    ρ⁴ = ρ³ * ρ

    r₁₁ = (η² + ζ²) / r³
    r₁₂ = -ξ * η / r³
    r₁₃ = -ξ * ζ / r³
    r₂₁ = r₁₂
    r₂₂ = (ξ² + ζ²) / r³
    r₂₃ = -η * ζ / r³
    r₃₁ = r₁₃
    r₃₂ = r₂₃
    r₃₃ = ρ² / r³

    θ₁₁ = 2 * ξ * η / ρ⁴
    θ₁₂ = (η² - ξ²) / ρ⁴
    θ₁₃ = 0
    θ₂₁ = θ₁₂
    θ₂₂ = -θ₁₁
    θ₂₃ = 0
    θ₃₁ = θ₁₃
    θ₃₂ = θ₂₃
    θ₃₃ = 0

    ϕ₁₁ = ζ * (-ξ² * ζ² - 3 * ξ² * ρ² + ρ² * r²) / (ρ³ * r⁴)
    ϕ₁₂ = -ξ * η * ζ * (3 * ξ² + 3 * η² + ζ²) / (ρ³ * r⁴)
    ϕ₁₃ = ξ * (ρ² - ζ²) / (ρ * r⁴)
    ϕ₂₁ = ϕ₁₂
    ϕ₂₂ = ζ * (-η² * ζ² - 3 * η² * ρ² + ρ² * r²) / (ρ³ * r⁴)
    ϕ₂₃ = η * (ρ² - ζ²) / (ρ * r⁴)
    ϕ₃₁ = ϕ₁₃
    ϕ₃₂ = ϕ₂₃
    ϕ₃₃ = 2ζ * ρ / r⁴

    Hu₁ = reshape([r₁₁ r₁₂ r₁₃;
            r₂₁ r₂₂ r₂₃;
            r₃₁ r₃₂ r₃₃], 1, 3, 3)

    Hu₂ = reshape([θ₁₁ θ₁₂ θ₁₃;
            θ₂₁ θ₂₂ θ₂₃;
            θ₃₁ θ₃₂ θ₃₃], 1, 3, 3)

    Hu₃ = reshape([ϕ₁₁ ϕ₁₂ ϕ₁₃;
            ϕ₂₁ ϕ₂₂ ϕ₂₃;
            ϕ₃₁ ϕ₃₂ ϕ₃₃], 1, 3, 3)

    hess_u = vcat(Hu₁, Hu₂, Hu₃)

    return SArray{Tuple{3,3,3},Float64}(hess_u)
end

lb = SA[0.1, 0.2, 0.3]
ub = SA[2.2, 1.8, 1.6]
uc = chebpoints((50, 50, 50), lb, ub)
cb = chebinterp(f.(uc), lb, ub)
cs = ChebyshevSeries(cb.coefs, lb, ub)
ts = TransformedChebyshevSeries(cs, u, ∇u, Hu)
print(ts)
ts

3-D transformed Chebyshev series of order (26, 13, 16)

3-dimensional transformed Chebyshev series of order (26, 13, 16) for u(x) ∈ [0.100, 2.200]×[0.200, 1.800]×[0.300, 1.600]

In [9]:
f(x) = exp(cos(0.5x))

lb1, ub1 = SA[0.0], SA[0.5]
lb2, ub2 = SA[0.5], SA[1.0]

xc1 = chebpoints(50, lb1[], ub1[])
cb1 = chebinterp(f.(xc1), lb1[], ub1[])
cs1 = ChebyshevSeries(cb1.coefs, lb1, ub1)

xc2 = chebpoints(50, lb2[], ub2[])
cb2 = chebinterp(f.(xc2), lb2[], ub2[])
cs2 = ChebyshevSeries(cb2.coefs, lb2, ub2)

cc = ChebyshevCluster(cs1, cs2)
print(cc)
cc

Cluster of 2 1-D Chebyshev series

Cluster of 2 1-D Chebyshev series: 
1. Order 10, x ∈ [0.000, 0.500]
2. Order 10, x ∈ [0.500, 1.000]

In [10]:
cs1 = ChebyshevSeries(zeros(2, 2), SA[0.0, 1.0], SA[1.0, 2.0])
cs2 = ChebyshevSeries(zeros(2, 2), SA[1.0, 2.0], SA[2.0, 3.0])
cc = ChebyshevCluster(cs1, cs2)

print(cc)
cc

Cluster of 2 2-D Chebyshev series

Cluster of 2 2-D Chebyshev series: 
1. Order (1, 1), x ∈ [0.000, 1.000]×[1.000, 2.000]
2. Order (1, 1), x ∈ [1.000, 2.000]×[2.000, 3.000]